# <span style = 'color:Orange'>Ridge Regression</span>
> Multivariate data

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

In [2]:
df = pd.read_csv('ad.csv')
df.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


In [3]:
x = df.iloc[:, 0:-1]
y = df.iloc[:, -1]

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

### Hyperparameter Tuning

In [4]:
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000],
    'fit_intercept': [True, False],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga'],
    'positive': [True, False],
    'tol': [1e-5, 1e-4, 1e-3],
}

grid = GridSearchCV(estimator=Ridge(),
                    param_grid=param_grid,
                    cv = 5,
                    scoring='r2',
                    n_jobs=-1)

grid.fit(x_train, y_train)
print('Best paraemters: ',grid.best_params_)

print('Best model : ',grid.best_estimator_)
print('R2 score : ',grid.best_score_)

Best paraemters:  {'alpha': 0.001, 'fit_intercept': True, 'positive': False, 'solver': 'sag', 'tol': 0.001}
Best model :  Ridge(alpha=0.001, solver='sag', tol=0.001)
R2 score :  0.8880479650594209


/Users/lucifer/Library/Python/3.9/lib/python/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
1200 fits failed out of a total of 2880.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
240 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/lucifer/Library/Python/3.9/lib/python/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/lucifer/Library/Python/3.9/lib/python/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Users/lucifer/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_ridge.py", line 1249, in fi

In [5]:
# model training with best hyper parameters
rid = grid.best_estimator_
rid

Ridge(alpha=0.001, solver='sag', tol=0.001)

In [6]:
y_Pred = rid.predict(x_test)

print('Intercept :', rid.intercept_)
print('Coef :', rid.coef_)
print('r2_score : ',r2_score(y_test, y_Pred))

Intercept : 4.7080615751782595
Coef : [0.0545447  0.10090801 0.00439022]
r2_score :  0.9058662640633134


In [7]:
print('CV R2:', grid.best_score_)
print('Test R2:', r2_score(y_test, y_Pred))

CV R2: 0.8880479650594209
Test R2: 0.9058662640633134


In [15]:
# correlation matrix
df.corr()

,TV,Radio,Newspaper,Sales
TV,1.000000,0.054809,0.056648,0.901208
Radio,0.054809,1.000000,0.354104,0.349631
Newspaper,0.056648,0.354104,1.000000,0.157960
Sales,0.901208,0.349631,0.157960,1.000000


In [19]:
x_range = np.linspace(x['TV'].min(),  x['TV'].max())
y_range = np.linspace(x['Radio'].min(), x['Radio'].max())

xx,yy = np.meshgrid(x_range, y_range)
news_mean = df['Newspaper'].mean()
news_col = np.full(xx.ravel().shape, news_mean)
pred_ip = np.c_[xx.ravel(), yy.ravel(), news_col]

zz = rid.predict(pred_ip).reshape(xx.shape)



fig = px.scatter_3d(df, x = df['TV'],
                    y = df['Radio'],
                    z = df['Sales'],
                    color=df['Sales'],
                    color_continuous_scale='thermal',
                    title='True data')
fig.add_surface(x = x_range, y = y_range, z = zz, colorscale='thermal')

fig.show()

/Users/lucifer/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but Ridge was fitted with feature names

